In [2]:
# ============================================================
# Cell 1: Environment & Configuration
# ============================================================

from pathlib import Path

import polars as pl

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"

OBSERVATION_DIR = PROCESSED_DIR / "observations"

OUTPUT_FILE = (
    OBSERVATION_DIR
    / "master_observations.parquet"
)

# ------------------------------------------------------------------
# Canonical schema
# ------------------------------------------------------------------

REQUIRED_COLUMNS = (
    "timestamp",
    "h3_id",
    "variable",
    "value",
    "source",
)

# ------------------------------------------------------------------
# Discover observation files
# ------------------------------------------------------------------

observation_files = sorted(
    OBSERVATION_DIR.glob("observations_*.parquet")
)

print("=" * 60)
print("MASTER OBSERVATION BUILD")
print("=" * 60)
print(f"Observation files : {len(observation_files)}")

for file in observation_files:
    print(f" • {file.name}")

print(f"\nOutput : {OUTPUT_FILE}")
print("=" * 60)

MASTER OBSERVATION BUILD
Observation files : 1
 • observations_chirps_monthly.parquet

Output : c:\Projects\ews\zoryn\data\processed\observations\master_observations.parquet


In [3]:
# ============================================================
# Cell 2: Validate observation files
# ============================================================

validated_files = []

for file in observation_files:

    df = pl.scan_parquet(file)

    columns = tuple(df.collect_schema().names())

    if columns != REQUIRED_COLUMNS:
        raise ValueError(
            f"{file.name}\n"
            f"Expected : {REQUIRED_COLUMNS}\n"
            f"Found    : {columns}"
        )

    row_count = (
        df.select(pl.len())
        .collect()
        .item()
    )

    validated_files.append(file)

    print(
        f"[OK] {file.name:<40}"
        f"{row_count:>12,} rows"
    )

if not validated_files:
    raise ValueError(
        "No valid observation files were found."
    )

print("=" * 60)
print(f"Validated files : {len(validated_files)}")
print("=" * 60)

[OK] observations_chirps_monthly.parquet       11,680,044 rows
Validated files : 1


In [4]:
# ============================================================
# Cell 3: Build master observation dataset
# ============================================================

lazy_frames = [
    pl.scan_parquet(file)
    for file in validated_files
]

master = (
    pl.concat(
        lazy_frames,
        how="vertical",
    )
    .sort(
        by=[
            "timestamp",
            "h3_id",
            "variable",
        ]
    )
)

print("=" * 60)
print("MASTER OBSERVATIONS")
print("=" * 60)

print(
    master.select(
        [
            pl.len().alias("rows"),
            pl.col("timestamp").n_unique().alias("timestamps"),
            pl.col("h3_id").n_unique().alias("h3_cells"),
            pl.col("variable").n_unique().alias("variables"),
            pl.col("source").n_unique().alias("sources"),
        ]
    ).collect()
)

print("=" * 60)

MASTER OBSERVATIONS
shape: (1, 5)
┌──────────┬────────────┬──────────┬───────────┬─────────┐
│ rows     ┆ timestamps ┆ h3_cells ┆ variables ┆ sources │
│ ---      ┆ ---        ┆ ---      ┆ ---       ┆ ---     │
│ u32      ┆ u32        ┆ u32      ┆ u32       ┆ u32     │
╞══════════╪════════════╪══════════╪═══════════╪═════════╡
│ 11680044 ┆ 276        ┆ 42319    ┆ 1         ┆ 1       │
└──────────┴────────────┴──────────┴───────────┴─────────┘


In [5]:
# ============================================================
# Cell 4: Write master observation dataset
# ============================================================

# Remove invalid observations before writing
master = master.filter(
    pl.col("value").is_finite()
)

master.sink_parquet(
    OUTPUT_FILE,
    compression="zstd",
)

master_df = pl.scan_parquet(OUTPUT_FILE)

summary = (
    master_df.select(
        [
            pl.len().alias("rows"),
            pl.col("timestamp").min().alias("first_timestamp"),
            pl.col("timestamp").max().alias("last_timestamp"),
            pl.col("h3_id").n_unique().alias("h3_cells"),
            pl.col("variable").n_unique().alias("variables"),
            pl.col("source").n_unique().alias("sources"),
        ]
    )
    .collect()
)

print("=" * 60)
print("MASTER OBSERVATIONS WRITTEN")
print("=" * 60)
print(summary)
print(f"\nSaved to:\n{OUTPUT_FILE}")
print("=" * 60)

MASTER OBSERVATIONS WRITTEN
shape: (1, 6)
┌──────────┬─────────────────┬────────────────┬──────────┬───────────┬─────────┐
│ rows     ┆ first_timestamp ┆ last_timestamp ┆ h3_cells ┆ variables ┆ sources │
│ ---      ┆ ---             ┆ ---            ┆ ---      ┆ ---       ┆ ---     │
│ u32      ┆ str             ┆ str            ┆ u32      ┆ u32       ┆ u32     │
╞══════════╪═════════════════╪════════════════╪══════════╪═══════════╪═════════╡
│ 11679216 ┆ 2002-01         ┆ 2024-12        ┆ 42316    ┆ 1         ┆ 1       │
└──────────┴─────────────────┴────────────────┴──────────┴───────────┴─────────┘

Saved to:
c:\Projects\ews\zoryn\data\processed\observations\master_observations.parquet
